# DT401 Week 4 Section 2: Data Relationships

**Module:** Data Fundamentals  
**Week:** 4 of 7  
**Section:** 2 of 3  
**Estimated Time:** 50 minutes

---

## Learning Objective

Explain how data relates within and across measures to inform workplace analysis.

By the end of this notebook, you will be able to:
- Distinguish between samples and populations
- Identify sources of sampling error and bias
- Calculate and interpret correlations between variables
- Use relationships to inform business decisions

---

## Context Recap

In Section 1, you learned how to gather data using quantitative and qualitative methods. Now we examine how data relates—how variables connect, how samples represent populations, and how correlation reveals patterns. Understanding relationships is essential for your Module Challenge Part A, where you'll analyze how workplace variables influence each other.

---

## How to Use This Notebook

This notebook contains a mix of instructional text, Python code cells, and interactive exercises. Follow the instructions in each section, run the code cells to see outputs, and complete the exercises to reinforce your learning.

You will find several types of boxes throughout the notebook to guide your learning:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b> <br>
These boxes contain important reminders about how this section relates to the overall module and your learning journey.
</div>

<div class="alert alert-block alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the topic discussed.
</div>

<div class="alert alert-block alert-success">
<b>Task boxes:</b> <br>
These have questions or tasks you should complete before progressing.

These may include hidden hints or solutions.
<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Hints will appear like this when you need guidance on tasks.

</details>

</div>

---

## Part 1: Understanding Samples and Populations

### Visual Overview

```mermaid
flowchart TD
    A[Population] --> B{Can we measure everyone?}
    B -->|Yes - Rare| C[Use full population]
    B -->|No - Common| D[Take a sample]
    D --> E{Is sample representative?}
    E -->|Yes| F[Reliable inference]
    E -->|No| G[Biased inference]
    G --> H[Coverage error, sampling error, non-response]
```

**What this diagram shows:**
We rarely have access to complete populations, so we sample. A representative sample allows reliable inference about the population; biased samples mislead.

**Why this matters in your role:**
A retail analyst surveying customer satisfaction during weekday mornings misses working professionals. Conclusions about "customer satisfaction" are biased—they only represent daytime shoppers, not the full customer population.

---

### Worked Example: Analyzing Temperature Samples

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load weather dataset
weather_df = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, low_memory=False)
weather_df.columns = [c.split(' ')[0].strip() for c in weather_df.columns]
weather_df = weather_df.rename(columns={
    'time': 'date',
    'temperature_2m': 'temperature',
    'wind_speed_10m': 'wind_speed',
    'relative_humidity_2m': 'relative_humidity'
})
weather_df = weather_df[weather_df['date'].str.match(r'^\d{4}-\d{2}-\d{2}T', na=False)].reset_index(drop=True)
for col in weather_df.columns[1:]:
    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')
weather_df['date'] = pd.to_datetime(weather_df['date'])

# Full population: all temperature readings
population = weather_df['temperature'].dropna()
print(f"Population size: {len(population)} observations")
print(f"Population mean: {population.mean():.2f}°C")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We loaded the complete weather dataset (our "population" for this exercise)
- We calculated the mean temperature across all observations
- This represents the true population parameter we're trying to estimate

**Key lesson:** In real scenarios, we rarely have full populations. Here we do, which lets us see how samples compare to reality.

</div>

Now let's take random samples and see how well they estimate the population:

In [ ]:
# Take 5 random samples of 100 observations each
np.random.seed(42)  # For reproducibility
sample_means = []

for i in range(5):
    sample = population.sample(n=100)
    sample_mean = sample.mean()
    sample_means.append(sample_mean)
    print(f"Sample {i+1} mean: {sample_mean:.2f}°C")

print(f"\nPopulation mean: {population.mean():.2f}°C")
print(f"Average of sample means: {np.mean(sample_means):.2f}°C")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We took 5 different random samples of 100 observations
- Each sample produced a slightly different mean
- None exactly matched the population mean, but all were close
- Averaged together, they get even closer

**Key lesson:** Random sampling error is unavoidable, but it's not bias—errors cancel out across multiple samples. Larger samples generally produce more accurate estimates.

</div>

### Common Sampling Problems

Now let's see what happens with **biased sampling:**

In [ ]:
# Biased sample: only summer months
weather_df['month'] = weather_df['date'].dt.month
summer_sample = weather_df[weather_df['month'].isin([6,7,8])]
summer_mean = summer_sample['temperature'].mean()

print(f"Population mean (all months): {population.mean():.2f}°C")
print(f"Biased sample mean (summer only): {summer_mean:.2f}°C")
print(f"Bias: {summer_mean - population.mean():.2f}°C")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We created a biased sample by only including summer months
- The sample mean is much higher than the population mean
- This bias won't cancel out—it's systematic distortion

**Key lesson:** Sampling bias produces consistently wrong estimates. A council planning winter services based on average annual temperature (inflated by summer readings) would underestimate gritting needs.

</div>

**Business Impact Example - Healthcare:**

A hospital surveys patient satisfaction by emailing patients. **Bias:** Older patients less comfortable with email are underrepresented. If they have different satisfaction levels (perhaps needing more in-person support), the survey produces biased results.

**Solution:** Use multiple channels (email, phone, postal) to ensure representative coverage.

---

## Part 2: Modified Task - Identifying Sampling Bias

---

### Modified Task: Detecting Bias in Samples

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Let's explore how different sampling strategies affect results. We'll compare weekend temperatures vs. weekday temperatures.

Complete the code below to:
1. Calculate mean temperature for weekdays only
2. Calculate mean temperature for weekends only
3. Compare both to the full population mean

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Use `weather_df['date'].dt.dayofweek` to get the day number (0=Monday, 6=Sunday).
- Weekdays: day < 5
- Weekends: day >= 5

Filter the dataframe, then calculate `.mean()` on the temperature column.

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Add day of week column
weather_df['dayofweek'] = weather_df['date'].dt.dayofweek

# Weekday sample (Monday-Friday, 0-4)
weekday_sample = weather_df[weather_df['dayofweek'] < 5]
weekday_mean = weekday_sample['temperature'].mean()

# Weekend sample (Saturday-Sunday, 5-6)
weekend_sample = weather_df[weather_df['dayofweek'] >= 5]
weekend_mean = weekend_sample['temperature'].mean()

# Compare to population
print(f"Population mean: {population.mean():.2f}°C")
print(f"Weekday mean: {weekday_mean:.2f}°C (difference: {weekday_mean - population.mean():.2f}°C)")
print(f"Weekend mean: {weekend_mean:.2f}°C (difference: {weekend_mean - population.mean():.2f}°C)")
```

</details>

</div>

In [ ]:
# Add day of week column
weather_df['dayofweek'] = weather_df['date'].dt.dayofweek

# TODO: Calculate weekday sample mean
# weekday_sample = ...
# weekday_mean = ...

# TODO: Calculate weekend sample mean
# weekend_sample = ...
# weekend_mean = ...

# TODO: Print comparison
# print(f"Population mean: {population.mean():.2f}°C")
# print(f"Weekday mean: ...")
# print(f"Weekend mean: ...")

**Expected output:** Weekday and weekend means should be very close to population mean (weather doesn't care about human calendar!) This shows that sampling days of the week isn't biased for temperature.

**Business question:** For what type of data *would* weekday vs. weekend sampling create bias?

**Answer:** Retail foot traffic, restaurant bookings, public transport usage, gym attendance—anything driven by human behavior patterns.

---

## Part 3: Understanding Correlation

### Visual Overview

```mermaid
flowchart LR
    A[Two Variables] --> B{How do they relate?}
    B --> C[Positive Correlation]
    B --> D[Negative Correlation]
    B --> E[No Correlation]
    C --> F[As X increases, Y increases]
    D --> G[As X increases, Y decreases]
    E --> H[X and Y are independent]
```

**What this diagram shows:**
Correlation quantifies how two variables move together. Positive correlation means they move in the same direction, negative means opposite directions, and zero means they're independent.

**Why this matters in your role:**
In finance, understanding that loan default risk correlates negatively with income helps set lending criteria. In retail, knowing that ice cream sales correlate positively with temperature informs inventory planning.

---

### Worked Example: Temperature and Precipitation

In [ ]:
# Calculate correlation between temperature and precipitation
clean_data = weather_df[['temperature', 'precipitation']].dropna()
correlation = clean_data['temperature'].corr(clean_data['precipitation'])

print(f"Correlation: {correlation:.3f}")

# Visualize relationship
plt.figure(figsize=(10, 6))
plt.scatter(clean_data['temperature'], clean_data['precipitation'], 
            alpha=0.3, s=10)
plt.xlabel('Temperature (°C)')
plt.ylabel('Precipitation (mm)')
plt.title('Temperature vs Precipitation')
plt.show()

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

- We calculated the correlation coefficient between temperature and precipitation
- A value near 0 suggests weak or no relationship
- The scatterplot visually confirms: points spread across the plot
- Temperature and precipitation are largely independent in London

**Key lesson:** Correlation ranges from -1 (perfect negative) through 0 (no relationship) to +1 (perfect positive). Values near 0 indicate independence.

</div>

**Business Interpretation:**

A garden center manager might assume warm weather means less rain (good for sales). However, the weak correlation shows this isn't reliable in London—warm days can still be rainy. Better to track actual rainfall, not infer it from temperature.

### Finding Stronger Correlations

In [ ]:
# Check correlation between temperature and relative humidity
if 'relative_humidity' in weather_df.columns:
    humidity_corr = weather_df['temperature'].corr(
        weather_df['relative_humidity']
    )
    print(f"Temperature vs Humidity correlation: {humidity_corr:.3f}")

# Check correlation between wind speed and precipitation
if 'wind_speed' in weather_df.columns:
    wind_corr = weather_df['wind_speed'].corr(
        weather_df['precipitation']
    )
    print(f"Wind Speed vs Precipitation correlation: {wind_corr:.3f}")

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
Correlation does NOT imply causation. Temperature and humidity may correlate, but this doesn't mean one causes the other—both are influenced by weather systems.
</div>

---

## Part 4: Interactive Practice - Correlation Analysis

Let's practice calculating and interpreting correlations with workplace scenarios.

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Create a correlation matrix showing relationships between multiple weather variables.

Complete the code to:
1. Select 4 numerical columns from the weather data
2. Calculate correlations between all pairs
3. Display the correlation matrix

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

Use `.corr()` on a dataframe with multiple columns to get a correlation matrix.

Example:
```python
selected_vars = weather_df[['temperature', 'precipitation', 'wind_speed', 'pressure']]
correlation_matrix = selected_vars.corr()
print(correlation_matrix)
```

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for the solution</summary>

```python
# Select variables (adjust column names if needed)
variables = ['temperature', 'precipitation', 'wind_speed', 'pressure']
available_vars = [v for v in variables if v in weather_df.columns]

# Calculate correlation matrix
selected_data = weather_df[available_vars].dropna()
corr_matrix = selected_data.corr()

print("Correlation Matrix:")
print(corr_matrix.round(3))

# Visualize with heatmap
import seaborn as sns
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Weather Variables Correlation Matrix')
plt.show()
```

</details>

</div>

In [ ]:
# TODO: Select numerical columns
# variables = ['temperature', 'precipitation', ...]

# TODO: Calculate correlation matrix
# selected_data = weather_df[variables].dropna()
# corr_matrix = ...

# TODO: Display results
# print(corr_matrix.round(3))

**Interpretation Questions:**

1. Which pairs of variables have the strongest positive correlation?
2. Which pairs have the strongest negative correlation?
3. Which variables appear mostly independent (correlation near 0)?
4. How would you explain these relationships to a non-technical stakeholder?

---

## Part 5: Independent Task - Workplace Correlation Analysis

### Your Challenge

Apply correlation analysis to a workplace scenario. Choose from:

**Healthcare:** Analyze correlations between patient age, length of stay, treatment cost, and readmission risk.

**Retail:** Examine correlations between product price, sales volume, customer ratings, and return rates.

**Local Government:** Investigate correlations between service request volume, response time, citizen satisfaction, and repeat requests.

**Finance:** Explore correlations between account balance, transaction frequency, customer service calls, and customer tenure.

**Charity:** Study correlations between donor age, donation frequency, donation amount, and engagement with communications.

**Your task:** 
1. Identify 4 variables from your workplace that might correlate
2. Hypothesize the relationships (positive, negative, none)
3. Explain what strong correlations would mean for business decisions
4. Describe how you would gather data to test these hypotheses

<div class="alert alert-block alert-success">
<b>Your turn:</b> <br>
Design a correlation analysis plan for your workplace.

<details>
    <summary style="color:black;font-weight:bold">Click here for a hint</summary>

**Structure your analysis:**
1. Choose variables that are numerical and logically connected
2. Consider both direct relationships (price → sales) and indirect (satisfaction → loyalty → sales)
3. Think about what data sources exist or could be created
4. Consider business actions: which correlations would inform specific decisions?

</details>

<details>
    <summary style="color:green;font-weight:bold">Click here for an example solution</summary>

**Scenario:** Retail - Online clothing store

**Variables identified:**
1. Product price (£)
2. Number of customer reviews
3. Average star rating (1-5)
4. Return rate (%)

**Hypothesized relationships:**

| Variable Pair | Expected Correlation | Reasoning |
|:--------------|:---------------------|:----------|
| Price vs Reviews | Positive (moderate, +0.4) | More expensive items get more attention, people invest time in reviews |
| Reviews vs Rating | Weak (+0.2) | More reviews might mean more diverse opinions, averaging out |
| Rating vs Return Rate | Negative (strong, -0.7) | Better ratings mean fewer disappointments, fewer returns |
| Price vs Return Rate | Positive (weak, +0.3) | Higher expectations for expensive items, slight increase in returns |

**Business implications:**

**If Rating vs Return Rate is strongly negative (-0.7):**
- **Decision:** Prioritize product quality improvements for low-rated items
- **Action:** Target items with <4.0 stars for quality audits
- **Expected impact:** Each 0.5 star improvement could reduce returns by 10%

**If Price vs Reviews is moderately positive (+0.4):**
- **Decision:** Encourage reviews on high-value items
- **Action:** Send review requests specifically for purchases over £50
- **Expected impact:** More social proof for profitable products

**If Price vs Return Rate is weak (+0.3):**
- **Insight:** Price itself isn't driving returns (quality matters more)
- **Decision:** Premium pricing is acceptable if quality is high
- **Action:** Focus on product descriptions and photos, not price reductions

**Data gathering plan:**
- **Source:** E-commerce platform database
- **Time period:** Last 12 months
- **Sample size:** All products with ≥20 sales (ensures statistical reliability)
- **Exclusions:** Seasonal items, discontinued products
- **Analysis:** Calculate correlations, create scatterplots, segment by product category
- **Validation:** Compare findings across different product categories to ensure patterns hold

</details>

</div>

**Write your correlation analysis plan below:**

**Scenario chosen:**


**Variables (4):**
1. 
2. 
3. 
4. 

**Hypothesized correlations:**

| Variable Pair | Expected Correlation | Reasoning |
|:--------------|:---------------------|:----------|
| | | |
| | | |
| | | |

**Business implications:**


**Data gathering approach:**


**Success criteria:**
- Four relevant numerical variables identified
- Clear hypotheses about correlation direction and strength
- Specific business decisions connected to each correlation
- Realistic data gathering plan
- Consideration of confounding factors

**Portfolio note:** This correlation analysis plan contributes to your Module Challenge Part A portfolio. It demonstrates understanding of relationships between variables and how they inform workplace decisions.

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO3:** *Select and apply appropriate data-gathering methods for a given data analysis scenario*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Identify two variables in your work that you suspect correlate. How would understanding this correlation, and any potential underlying causation, change any business decisions?

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 4 Section 2.** You can now:

- Compared five random samples of 100 temperature readings against the full population mean, then built a biased sample (summer months only) to see how sampling bias differs from random sampling error
- Calculated weekday and weekend temperature means and compared them against the population mean to test whether day-of-week sampling introduces bias
- Calculated the correlation coefficient between temperature and precipitation and visualised it on a scatterplot, then checked further pairwise correlations including temperature versus humidity
- Built a correlation matrix across four weather variables (temperature, precipitation, wind speed, and pressure) and interpreted the strongest and weakest relationships
- Designed a workplace correlation analysis plan identifying four variables, hypothesising the direction of each relationship, and connecting the findings to business decisions

**Key takeaways:**

- Random sampling error varies from sample to sample and cancels out on average. Sampling bias is systematic and does not cancel out, however large the sample
- Correlation does not imply causation. Two variables can move together because both are driven by a third factor

**What comes next:**

- **Section 3** turns to summarising and visualising data: measures of central tendency and dispersion, building on the mean and standard deviation calculations used here to compare samples

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 4 Section 2 Data Relationships"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.4 component on Aptem
5. Open `DT401_Week4_Section3.ipynb` when ready to continue